In [20]:
import pandas as pd
from pathlib import Path
import numpy as np

datos = pd.read_csv("../data/preprocessed/datos_2021.csv")

In [21]:
datos = datos.loc[
    datos["ClasificacionRecuperacion"] != "No recuperable"
].reset_index(drop=True)

In [22]:
filas_cero = datos.loc[
    datos["VentasNetas"].eq(0),
    ["IdRegistro", "Factura", "NumeroOT", "FechaFactura",
     "VentasNetas", "Venta$", "VentaTotal$", "Utilidad$"]
]

print("Filas con VentasNetas = 0:", len(filas_cero))
display(filas_cero.head(20))

Filas con VentasNetas = 0: 20353


,IdRegistro,Factura,NumeroOT,FechaFactura,VentasNetas,Venta$,VentaTotal$,Utilidad$
3,995441,53848,59935,2021-03-31,0,0.0,0.00,0.0
24,1250829,18418,38701,2021-03-31,0,0.0,1.00,0.0
25,1250828,18417,38700,2021-03-31,0,0.0,1.00,0.0
26,1250827,18420,38632,2021-03-31,0,0.0,154.59,0.0
33,1250859,31149,38655,2021-03-31,0,0.0,1589.71,0.0
43,1318728,114230,41101,2021-03-31,0,0.0,9510.97,0.0
75,1318648,53782,40405,2021-03-31,0,0.0,0.00,0.0
80,1318643,53790,40195,2021-03-31,0,0.0,0.00,0.0
124,291387,301349,33364,2021-03-31,0,0.0,964.48,0.0
126,291389,301342,33343,2021-03-31,0,0.0,0.00,0.0


In [23]:

columnas_fecha = ["FechaApertura", "FechaFactura"]
datos[columnas_fecha] = datos[columnas_fecha].replace(
    r"^\s*$", pd.NA, regex=True
)
datos["Dias"] = datos["Dias"].replace(r"^\s*$", pd.NA, regex=True)


for columna in columnas_fecha:
    datos[columna] = pd.to_datetime(
        datos[columna],
        format="mixed",
        dayfirst=True,
        errors="coerce"
    )

datos["Dias"] = pd.to_numeric(datos["Dias"], errors="coerce")


falta_apertura = datos["FechaApertura"].isna()
sin_dias = datos["Dias"].isna()
sin_fecha_factura = datos["FechaFactura"].isna()

eliminar = falta_apertura & (sin_dias | sin_fecha_factura)
cantidad_eliminada = int(eliminar.sum())
datos = datos.loc[~eliminar].copy()


imputar_apertura = datos["FechaApertura"].isna()
cantidad_imputada = int(imputar_apertura.sum())

datos.loc[imputar_apertura, "FechaApertura"] = (
    datos.loc[imputar_apertura, "FechaFactura"]
    - pd.to_timedelta(datos.loc[imputar_apertura, "Dias"], unit="D")
)

print(f"Entradas eliminadas: {cantidad_eliminada}")
print(f"FechaApertura imputadas: {cantidad_imputada}")

Entradas eliminadas: 9713
FechaApertura imputadas: 973


In [24]:
claves = [
    "FechaFactura", "Client", "Branch", "Vin",
    "FechaApertura", "FechaEntrega", "Taller",
    "TipoOrden", "NumeroOT"
]

columnas_valores = [
    "Venta$", "Costo$", "Utilidad$", "VentasNetas",
    "VentaMO$", "CostoMO$", "VentaMateriales$",
    "VentaPartes$", "CostoPartes$", "VentaTotal$", "CostoTotal$"
]


for columna in ["FechaFactura", "FechaApertura", "FechaEntrega"]:
    datos[columna] = pd.to_datetime(
        datos[columna],
        format="mixed",
        dayfirst=True,
        errors="coerce"
    )


valores = pd.DataFrame(index=datos.index)

for columna in columnas_valores:
    texto = (
        datos[columna]
        .astype("string")
        .str.replace(r"[$,]", "", regex=True)
        .str.strip()
        .replace("", pd.NA)
    )

    numero = pd.to_numeric(texto, errors="coerce")

    if (texto.notna() & numero.isna()).any():
        raise ValueError(f"No se pudieron convertir algunos valores de {columna}")

    valores[columna] = numero

valores = valores.round(2)


firma_base = pd.concat(
    [
        datos[claves].reset_index(drop=True),
        valores.reset_index(drop=True)
    ],
    axis=1
)

firmas = pd.util.hash_pandas_object(firma_base, index=False)

firma_opuesta = firma_base.copy()
firma_opuesta[columnas_valores] = -firma_opuesta[columnas_valores]
firmas_opuestas = pd.util.hash_pandas_object(firma_opuesta, index=False)


importes_completos = valores.notna().all(axis=1).to_numpy()
hay_importe_no_cero = valores.fillna(0).ne(0).any(axis=1).to_numpy()


firmas_series = pd.Series(firmas.to_numpy())
conteos = firmas_series.value_counts()
orden_en_grupo = firmas_series.groupby(firmas_series, sort=False).cumcount().to_numpy()
cantidad_opuestas = (
    pd.Series(firmas_opuestas.to_numpy())
    .map(conteos)
    .fillna(0)
    .to_numpy()
)

es_pareja = (
    (orden_en_grupo < cantidad_opuestas)
    & importes_completos
    & hay_importe_no_cero
)


primer_importe_no_cero = (
    valores.replace(0, np.nan)
    .bfill(axis=1)
    .iloc[:, 0]
    .fillna(0)
    .to_numpy(dtype=float)
)

clasificacion = np.full(len(datos), "Registro de venta", dtype=object)
clasificacion[es_pareja & (primer_importe_no_cero > 0)] = "Registro Cancelado"
clasificacion[es_pareja & (primer_importe_no_cero < 0)] = "Registro de Cancelacion"

datos["TipoRegistro"] = clasificacion

print(datos["TipoRegistro"].value_counts())

TipoRegistro
Registro de venta          105585
Registro de Cancelacion      1101
Registro Cancelado           1101
Name: count, dtype: int64


In [25]:
carpeta_salida = Path("../data/processed")
carpeta_salida.mkdir(parents=True, exist_ok=True)

ruta_salida = carpeta_salida / "datos_procesados_2021.csv"
datos.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

print(f"Archivo guardado en: {ruta_salida}")

Archivo guardado en: ..\data\processed\datos_procesados_2021.csv
